In [1]:
import numpy as np
import gzip
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn

import seaborn as sns


# Cargamos el dataset MNIST y lo particionamos

In [2]:
def load_mnist_dataset(mnist_path):
    x_trainval = get_images(Path(mnist_path)/Path('train-images-idx3-ubyte.gz'))
    y_trainval = get_labels(Path(mnist_path)/Path('train-labels-idx1-ubyte.gz'))

    x_train = x_trainval[:50000]
    y_train = y_trainval[:50000]

    x_val = x_trainval[50000:]
    y_val = y_trainval[50000:]

    x_test = get_images(Path(mnist_path)/Path('t10k-images-idx3-ubyte.gz'))
    y_test = get_labels(Path(mnist_path)/Path('t10k-labels-idx1-ubyte.gz'))

    return x_train, y_train, x_val, y_val, x_test, y_test

def get_labels(path):
    with gzip.open(path, 'rb') as data:
        labels = data.read()[8:]
        return np.frombuffer(labels, dtype=np.uint8)

def get_images(path):
    with gzip.open(path, 'rb') as data:
        _ = int.from_bytes(data.read(4), 'big')
        num_images = int.from_bytes(data.read(4), 'big')
        rows = int.from_bytes(data.read(4), 'big')
        cols = int.from_bytes(data.read(4), 'big')
        images = data.read()
        return np.frombuffer(images, dtype=np.uint8).reshape((num_images, rows, cols))


In [3]:
x_train, y_train, x_val, y_val, x_test, y_test = load_mnist_dataset('datasets/mnist')

In [4]:
x_train = x_train.copy().reshape(50000, -1).astype(np.float32)
y_train = y_train.copy().reshape(50000, 1)

x_val = x_val.copy().reshape(10000, -1).astype(np.float32)
y_val = y_val.copy().reshape(10000, 1)

x_test = x_test.copy().reshape(10000, -1).astype(np.float32)
y_test = y_test.copy().reshape(10000, 1)

def scale(x_mean, x_std, x_data):
    return (x_data - x_mean) / x_std

x_mean = x_train.mean()
x_std = x_train.std()

x_train = scale(x_mean, x_std, x_train)
x_val = scale(x_mean, x_std, x_val)
x_test = scale(x_mean, x_std, x_test)

In [5]:
from torch.utils.data import DataLoader, TensorDataset

x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.squeeze(), dtype=torch.long)
x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.squeeze(), dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.squeeze(), dtype=torch.long)

dataset = TensorDataset(x_train_tensor, y_train_tensor)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Red FF

In [6]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [7]:
first_model = FeedForwardNN(input_size=784, hidden_size=32, output_size=10)
second_model = FeedForwardNN(input_size=784, hidden_size=32, output_size=10)
third_model = FeedForwardNN(input_size=784, hidden_size=500, output_size=10)

# Función de pérdida

In [8]:
loss_fn = nn.CrossEntropyLoss()

# Optimizador

In [9]:
import torch.optim as optim

first_optimizer = optim.Adam(first_model.parameters(), lr=0.001)
second_optimizer = optim.SGD(second_model.parameters(), lr=0.001)
third_optimizer = optim.Adam(third_model.parameters(), lr=0.0015)

# Entrenamiento

In [10]:
def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        logists = model(x)
        predicciones = torch.argmax(logists, dim=1)
        correctas = (predicciones == y).sum().item()
        total = y.shape[0]
    return correctas/total

In [11]:
from collections import deque # disclaimer: googleé este paquete porque hace más idiomático el uso del historial
def train(model, loss_fn, optimizer, num_epochs=64, paciencia=10, delta=0.005):
    historial_eval = deque(maxlen=paciencia+1)
    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in dataloader:
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch) # ¿CÓMO QUE PYTHON NO USA BLOCK-LEVEL SCOPING PARA LOS FOR LOOPS?
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            eval_logists = model(x_val_tensor)
            eval_loss = loss_fn(eval_logists, y_val_tensor)
            eval_acc = accuracy(model, x_val_tensor, y_val_tensor)
            historial_eval.append(eval_loss.item())
        
        print(f"Época {epoch+1}")
        print(f"Pérdida al entrenar: {loss.item():.4f}")
        print(f"Exactitud al evaluar: {eval_acc:.4f}")
        print(f"Pérdida al evaluar: {eval_loss:.4f}")
        
        if len(historial_eval) > paciencia and 0 <= (historial_eval[0] - historial_eval[-1]) < delta:
            print("Se nos acaba la paciencia.") # yo cuando python
            break

In [12]:
train(model=first_model, loss_fn=loss_fn, optimizer=first_optimizer)
train(model=second_model, loss_fn=loss_fn, optimizer=second_optimizer)
train(model=third_model, loss_fn=loss_fn, optimizer=third_optimizer, num_epochs=12)

Época 1
Pérdida al entrenar: 0.3406
Exactitud al evaluar: 0.9429
Pérdida al evaluar: 0.2009
Época 2
Pérdida al entrenar: 0.0147
Exactitud al evaluar: 0.9550
Pérdida al evaluar: 0.1568
Época 3
Pérdida al entrenar: 0.0170
Exactitud al evaluar: 0.9618
Pérdida al evaluar: 0.1353
Época 4
Pérdida al entrenar: 0.1550
Exactitud al evaluar: 0.9599
Pérdida al evaluar: 0.1367
Época 5
Pérdida al entrenar: 0.0045
Exactitud al evaluar: 0.9633
Pérdida al evaluar: 0.1307
Época 6
Pérdida al entrenar: 0.0672
Exactitud al evaluar: 0.9620
Pérdida al evaluar: 0.1289
Época 7
Pérdida al entrenar: 0.0330
Exactitud al evaluar: 0.9657
Pérdida al evaluar: 0.1234
Época 8
Pérdida al entrenar: 0.0029
Exactitud al evaluar: 0.9652
Pérdida al evaluar: 0.1233
Época 9
Pérdida al entrenar: 0.0295
Exactitud al evaluar: 0.9650
Pérdida al evaluar: 0.1301
Época 10
Pérdida al entrenar: 0.0113
Exactitud al evaluar: 0.9700
Pérdida al evaluar: 0.1130
Época 11
Pérdida al entrenar: 0.0048
Exactitud al evaluar: 0.9676
Pérdida al ev

# Evaluación

In [13]:
def eval(model):
    model.eval()
    with torch.no_grad():
        predictions = model(x_test_tensor)
        test_loss = loss_fn(predictions, y_test_tensor)
        print(f"Loss: {test_loss.item():.4f}")
        print("\n")

In [14]:
eval(model=first_model)
eval(model=second_model)
eval(model=third_model)

Loss: 0.2857


Loss: 0.1585


Loss: 0.1617


